# LoRA fine-tuning — Llama 3B on BANKING77

This notebook runs the same pipeline as `finetune/run_lora_finetune.py`, suitable for **AWS SageMaker** notebook instances or local GPU runs.

Prerequisites:
- Processed splits under `notebooks/data/processed/` (from `data_preprocess.ipynb` or `python finetune/data_preprocess.py --out notebooks/data/processed`)
- Hugging Face token with access to `meta-llama/Llama-3.2-3B` (`HF_TOKEN` in environment or `.ENV`)
- `pip install -r requirements-finetune.txt`
- Hyperparameters in `finetune/settings/config.json` (loaded via `FinetuneConfig.load()`)

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "finetune" / "settings" / "config.json"

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / ".ENV")

In [ ]:
from finetune.settings import FinetuneConfig
from finetune.train_lora import run_lora_finetune

config = FinetuneConfig.load(CONFIG_PATH)

# Optional runtime overrides (prefer editing config.json for persistent changes):
# config.training.max_train_samples = 256
# config.model.load_in_4bit = True
# config.training.num_train_epochs = 1

config

### Run pipeline

1. **Baseline evaluation** on validation/test (before LoRA weight updates)
2. **LoRA fine-tuning** with PEFT
3. **Post-training evaluation** and JSON comparison (`evaluation_comparison.json`)

In [ ]:
comparison = run_lora_finetune(config)
comparison

In [ ]:
import json
from IPython.display import Markdown, display

path = config.paths.output_dir / config.evaluation.comparison_filename
display(Markdown(f"Comparison saved to `{path}`"))
if path.exists():
    display(json.loads(path.read_text()))